In [1]:

# ========================
# PARTE 0: MONTAR DRIVE E INSTALAR DEPENDÊNCIAS
# ========================
from google.colab import drive
drive.mount('/content/drive')

!apt-get update > /dev/null
!apt install chromium-chromedriver -y > /dev/null || true
!cp /usr/lib/chromium-browser/chromedriver /usr/bin || true
!pip install selenium pandas openpyxl requests > /dev/null

# ========================
# PARTE 1: IMPORTS E CONFIGS
# ========================
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import pandas as pd
import os, time, requests, re
from datetime import datetime

options = Options()
options.binary_location = "/usr/bin/chromium-browser"
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--blink-settings=imagesEnabled=false")
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 7)

# ========================
# PARTE 2: CONFIG ALGOLIA
# ========================
ALGOLIA_APP_ID = "1CF3ZT43ZU"
ALGOLIA_API_KEY = "150c68d1c61fc1835826a57a203dab72"
ALGOLIA_INDEX = "production_products"

headers_algolia = {
    "Content-Type": "application/json",
    "X-Algolia-Application-Id": ALGOLIA_APP_ID,
    "X-Algolia-API-Key": ALGOLIA_API_KEY,
}

termos_busca = ["pisos", "revestimentos", "porcelanatos"]
modo = input("Deseja coletar 100% ou 50% dos produtos? (digite 100 ou 50): ").strip()
modo = 1.0 if modo == "100" else 0.5

# ========================
# PARTE 3: BUSCA INICIAL PELA API (Código, Nome, Preço, Imagem, URL)
# ========================
print("\n🔍 Coletando dados iniciais via API...")

def buscar_dados_iniciais(termos):
    todos_produtos = []
    for termo in termos:
        pagina = 0
        produtos_termo = []
        print(f"🔎 Buscando: {termo}")
        while True:
            url = f"https://{ALGOLIA_APP_ID}-dsn.algolia.net/1/indexes/{ALGOLIA_INDEX}/query"
            payload = { "params": f"query={termo}&page={pagina}&hitsPerPage=250" }
            res = requests.post(url, headers=headers_algolia, json=payload)
            if res.status_code != 200:
                break
            data = res.json()
            hits = data.get("hits", [])
            if not hits:
                break

            for item in hits:
                if item.get('url'):
                    produto = {
                        'Código': item.get('objectID'),
                        'Nome': item.get('name'),
                        'Preço Promocional': item.get('price') or item.get('price_sale') or 'N/A',
                        'Imagem': item.get('image', 'N/A'),
                        'URL': f"https://www.leroymerlin.com.br{item.get('url')}",
                        'Loja': "Leroy Merlin"
                    }
                    produtos_termo.append(produto)

            pagina += 1
            if pagina >= data.get("nbPages", 0):
                break

        limite = int(len(produtos_termo) * modo)
        print(f"🔗 {limite} produtos adicionados de {termo}")
        todos_produtos.extend(produtos_termo[:limite])
    return todos_produtos

todos_dados_iniciais = buscar_dados_iniciais(termos_busca)
print(f"✅ Total de produtos coletados da API: {len(todos_dados_iniciais)}\n")

# ========================
# PARTE 4: COMPLEMENTAR COM SELENIUM (Descrição e Especificações com tolerância a falhas)
# ========================
def extrair_dados_complementares(dados_iniciais):
    url = dados_iniciais['URL']
    try:
        driver.get(url)
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "body")))
    except Exception as e:
        print(f"⚠️ Erro ao acessar: {url}\n→ {e}")
        dados_iniciais['Descrição'] = "N/A"
        dados_iniciais['Especificações'] = "N/A"
        return dados_iniciais

    # Descrição
    try:
        meta_desc = driver.find_elements(By.CSS_SELECTOR, 'meta[name="description"]')
        dados_iniciais['Descrição'] = meta_desc[0].get_attribute("content").strip() if meta_desc else "N/A"
    except:
        dados_iniciais['Descrição'] = "N/A"

    # Especificações
    try:
        specs = {}
        linhas = driver.find_elements(By.CSS_SELECTOR, 'table tr')
        for tr in linhas:
            try:
                k = tr.find_element(By.CSS_SELECTOR, 'th').text.strip()
                v = tr.find_element(By.CSS_SELECTOR, 'td').text.strip()
                specs[k] = v
            except:
                continue
        dados_iniciais['Especificações'] = "\n".join(f"{k}: {v}" for k, v in specs.items()) if specs else "N/A"
    except:
        dados_iniciais['Especificações'] = "N/A"

    return dados_iniciais

# ========================
# PARTE 5: COLETA E SALVAMENTO A CADA 100
# ========================
data_nome = datetime.now().strftime("%Y-%m-%d")
base_drive = f"/content/drive/MyDrive/WebScraping/FEITO/Leroy/COLETA_LEROY/{data_nome}"
pasta_imagens = os.path.join(base_drive, "imagens")
os.makedirs(pasta_imagens, exist_ok=True)

todos = []
for i, produto in enumerate(todos_dados_iniciais, 1):
    try:
        dados = extrair_dados_complementares(produto)
        codigo = dados.get("Código", f"produto_{i}")
        caminho_img = os.path.join(pasta_imagens, f"{codigo}.png")

        if not os.path.exists(caminho_img) and dados.get("Imagem") not in [None, "N/A"]:
            img = requests.get(dados['Imagem'], stream=True, timeout=10)
            if img.status_code == 200:
                with open(caminho_img, 'wb') as f:
                    for chunk in img.iter_content(1024):
                        f.write(chunk)

        todos.append(dados)

        if i % 100 == 0 or i == len(todos_dados_iniciais):
            df = pd.DataFrame(todos)
            colunas = ['Código', 'Nome', 'Preço Promocional', 'Descrição', 'Especificações', 'Imagem', 'URL', 'Loja']
            df = df[colunas]
            df.to_csv(os.path.join(base_drive, f"leroymerlin_parcial_{i}.csv"), index=False, sep=";")
            df.to_excel(os.path.join(base_drive, f"leroymerlin_parcial_{i}.xlsx"), index=False)
            print(f"💾 Dados salvos com {i} produtos.")
            todos.clear()

    except Exception as e:
        print(f"❌ Erro inesperado no produto {i}:", e)

# ========================
# PARTE 6: UNIFICAR CSVs E Excel
# ========================
print("\n📁 Unificando arquivos CSV...")
csvs = [f for f in os.listdir(base_drive) if f.endswith(".csv") and "FINAL" not in f]
df_final = pd.DataFrame()

for csv in csvs:
    df = pd.read_csv(os.path.join(base_drive, csv), sep=";")
    df_final = pd.concat([df_final, df], ignore_index=True)

csv_final = os.path.join(base_drive, f"leroymerlin_FINAL_{data_nome}.csv")
excel_final = os.path.join(base_drive, f"leroymerlin_FINAL_{data_nome}.xlsx")

df_final.to_csv(csv_final, index=False, sep=";")
df_final.to_excel(excel_final, index=False)

print(f"✅ CSV final salvo: {csv_final}")
print(f"✅ Excel final salvo: {excel_final}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


cp: '/usr/lib/chromium-browser/chromedriver' and '/usr/bin/chromedriver' are the same file
Deseja coletar 100% ou 50% dos produtos? (digite 100 ou 50): 50

🔍 Coletando dados iniciais via API...
🔎 Buscando: pisos
🔗 500 produtos adicionados de pisos
🔎 Buscando: revestimentos
🔗 500 produtos adicionados de revestimentos
🔎 Buscando: porcelanatos
🔗 500 produtos adicionados de porcelanatos
✅ Total de produtos coletados da API: 1500

⚠️ Erro ao acessar: https://www.leroymerlin.com.brhttps://www.leroymerlin.com.br/piso-ceramico-marmorizado-brilhante-borda-arredondada-interno-61,5x61,5cm-brisa-rosagres_91700182
→ Message: unknown error: net::ERR_NAME_NOT_RESOLVED
  (Session info: c

KeyboardInterrupt: 

In [3]:
# ========================
# PARTE 0: MONTAR DRIVE E INSTALAR DEPENDÊNCIAS
# ========================
from google.colab import drive
drive.mount('/content/drive')

!apt-get update > /dev/null
!apt install chromium-chromedriver -y > /dev/null || true
!cp /usr/lib/chromium-browser/chromedriver /usr/bin || true
!pip install selenium pandas openpyxl requests > /dev/null

# ========================
# PARTE 1: IMPORTS E CONFIGS
# ========================
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import pandas as pd
import os, time, requests, re
from datetime import datetime

options = Options()
options.binary_location = "/usr/bin/chromium-browser"
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--blink-settings=imagesEnabled=false")
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 7)

# ========================
# PARTE 2: CONFIG ALGOLIA
# ========================
ALGOLIA_APP_ID = "1CF3ZT43ZU"
ALGOLIA_API_KEY = "150c68d1c61fc1835826a57a203dab72"
ALGOLIA_INDEX = "production_products"

headers_algolia = {
    "Content-Type": "application/json",
    "X-Algolia-Application-Id": ALGOLIA_APP_ID,
    "X-Algolia-API-Key": ALGOLIA_API_KEY,
}

termos_busca = ["piso laminado", "laminado", "durafloor"]
modo = input("Deseja coletar 100% ou 50% dos produtos? (digite 100 ou 50): ").strip()
modo = 1.0 if modo == "100" else 0.5

# ========================
# PARTE 3: BUSCA INICIAL VIA API
# ========================
print("\n🔍 Coletando produtos de Pisos Laminados via API...")

def buscar_dados_iniciais(termos):
    todos_produtos = []
    for termo in termos:
        pagina = 0
        produtos_termo = []
        print(f"🔎 Buscando: {termo}")
        while True:
            url = f"https://{ALGOLIA_APP_ID}-dsn.algolia.net/1/indexes/{ALGOLIA_INDEX}/query"
            payload = { "params": f"query={termo}&page={pagina}&hitsPerPage=250" }
            res = requests.post(url, headers=headers_algolia, json=payload)
            if res.status_code != 200:
                break
            data = res.json()
            hits = data.get("hits", [])
            if not hits:
                break
            for item in hits:
                if item.get('url'):
                    produto = {
                        'Código': item.get('objectID'),
                        'Nome': item.get('name'),
                        'Preço Promocional': item.get('price') or item.get('price_sale') or 'N/A',
                        'Imagem': item.get('image', 'N/A'),
                        'URL': f"https://www.leroymerlin.com.br{item.get('url')}",
                        'Loja': "Leroy Merlin"
                    }
                    produtos_termo.append(produto)
            pagina += 1
            if pagina >= data.get("nbPages", 0):
                break
        limite = int(len(produtos_termo) * modo)
        print(f"🔗 {limite} produtos adicionados de {termo}")
        todos_produtos.extend(produtos_termo[:limite])
    return todos_produtos

todos_dados_iniciais = buscar_dados_iniciais(termos_busca)
print(f"✅ Total de produtos coletados: {len(todos_dados_iniciais)}\n")

# ========================
# PARTE 4: COMPLETAR COM SELENIUM (Descrição + Especificações)
# ========================
def extrair_dados_complementares(dados_iniciais):
    url = dados_iniciais['URL']
    try:
        driver.get(url)
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "body")))
    except Exception as e:
        print(f"❌ Erro ao acessar o produto: {url}\n📄 Traceback: {e}")
        dados_iniciais['Descrição'] = "N/A"
        dados_iniciais['Especificações'] = "N/A"
        return dados_iniciais

    try:
        meta_desc = driver.find_elements(By.CSS_SELECTOR, 'meta[name="description"]')
        dados_iniciais['Descrição'] = meta_desc[0].get_attribute("content").strip() if meta_desc else "N/A"
    except:
        dados_iniciais['Descrição'] = "N/A"

    try:
        specs = {}
        linhas = driver.find_elements(By.CSS_SELECTOR, 'table tr')
        for tr in linhas:
            try:
                k = tr.find_element(By.CSS_SELECTOR, 'th').text.strip()
                v = tr.find_element(By.CSS_SELECTOR, 'td').text.strip()
                specs[k] = v
            except:
                continue
        dados_iniciais['Especificações'] = "\n".join(f"{k}: {v}" for k, v in specs.items()) if specs else "N/A"
    except:
        dados_iniciais['Especificações'] = "N/A"

    return dados_iniciais

# ========================
# PARTE 5: COLETA E SALVAMENTO
# ========================
data_nome = datetime.now().strftime("%Y-%m-%d")
base_drive = f"/content/drive/MyDrive/WebScraping/FEITO/Leroy/COLETA_LEROY/{data_nome}"
pasta_imagens = os.path.join(base_drive, "imagens")
os.makedirs(pasta_imagens, exist_ok=True)

todos = []
for i, produto in enumerate(todos_dados_iniciais, 1):
    try:
        dados = extrair_dados_complementares(produto)
        codigo = dados.get("Código", f"produto_{i}")
        caminho_img = os.path.join(pasta_imagens, f"{codigo}.png")

        if not os.path.exists(caminho_img) and dados.get("Imagem") not in [None, "N/A"]:
            img = requests.get(dados['Imagem'], stream=True, timeout=10)
            if img.status_code == 200:
                with open(caminho_img, 'wb') as f:
                    for chunk in img.iter_content(1024):
                        f.write(chunk)

        todos.append(dados)

        if i % 100 == 0 or i == len(todos_dados_iniciais):
            df = pd.DataFrame(todos)
            colunas = ['Código', 'Nome', 'Preço Promocional', 'Descrição', 'Especificações', 'Imagem', 'URL', 'Loja']
            df = df[colunas]
            df.to_csv(os.path.join(base_drive, f"leroymerlin_parcial_{i}.csv"), index=False, sep=";")
            df.to_excel(os.path.join(base_drive, f"leroymerlin_parcial_{i}.xlsx"), index=False)
            print(f"📂 Dados salvos com {i} produtos.")
            todos.clear()

    except Exception as e:
        print(f"❌ Erro inesperado no produto {i}: {e}")

# ========================
# PARTE 6: UNIFICAR
# ========================
print("\n📁 Unificando arquivos CSV...")
csvs = [f for f in os.listdir(base_drive) if f.endswith(".csv") and "FINAL" not in f]
df_final = pd.DataFrame()
for csv in csvs:
    df = pd.read_csv(os.path.join(base_drive, csv), sep=";")
    df_final = pd.concat([df_final, df], ignore_index=True)

csv_final = os.path.join(base_drive, f"leroymerlin_FINAL_{data_nome}.csv")
excel_final = os.path.join(base_drive, f"leroymerlin_FINAL_{data_nome}.xlsx")

df_final.to_csv(csv_final, index=False, sep=";")
df_final.to_excel(excel_final, index=False)

print(f"✅ CSV final salvo: {csv_final}")
print(f"✅ Excel final salvo: {excel_final}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


cp: '/usr/lib/chromium-browser/chromedriver' and '/usr/bin/chromedriver' are the same file
Deseja coletar 100% ou 50% dos produtos? (digite 100 ou 50): 100

🔍 Coletando produtos de Pisos Laminados via API...
🔎 Buscando: piso laminado
🔗 1000 produtos adicionados de piso laminado
🔎 Buscando: laminado
🔗 1000 produtos adicionados de laminado
🔎 Buscando: durafloor
🔗 140 produtos adicionados de durafloor
✅ Total de produtos coletados: 2140

❌ Erro ao acessar o produto: https://www.leroymerlin.com.brhttps://www.leroymerlin.com.br/piso-laminado-prime-cappuccino-136x19,7cm-m2-eucafloor_87744965
📄 Traceback: Message: unknown error: net::ERR_NAME_NOT_RESOLVED
  (Session info: chrome

KeyboardInterrupt: 